# Prototypical Networks — Prototypical Networks for Few-shot Learning (2017)

_Papers · Meta-learning, Bayesian & Robustness_

**Average each class's few support examples into one prototype point, then label a query by the nearest prototype.**

---

This notebook is a practice scaffold for the **Prototypical Networks — Prototypical Networks for Few-shot Learning (2017)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- 0. Sanity-check the worked example: prototype = mean, softmax over -d. ---
A = torch.tensor([[1., 1.], [3., 1.]]); B = torch.tensor([[4., 3.], [6., 3.]])
cA, cB = A.mean(0), B.mean(0)                  # Eqn. 1: prototypes (the means)
q = torch.tensor([3., 2.])
dA = ((q - cA)**2).sum(); dB = ((q - cB)**2).sum()   # squared Euclidean distance
p = torch.softmax(torch.tensor([-dA, -dB]), 0)       # Eqn. 2: softmax over NEGATIVE d
print("worked: cA", cA.tolist(), "cB", cB.tolist(), "dA", float(dA), "dB", float(dB))
print("worked: p(A)", round(float(p[0]), 4), "p(B)", round(float(p[1]), 4))
# worked: cA [2.0, 1.0] cB [5.0, 3.0] dA 2.0 dB 5.0
# worked: p(A) 0.9526 p(B) 0.0474


# --- 1. A synthetic few-shot dataset: many classes, disjoint train/test splits. ---
# Each class is a Gaussian blob whose class signal lives in a 2-d latent space, mapped
# into 20-d and buried in nuisance noise -- so a learned embedding helps.
RAWDIM, PER, LAT = 20, 40, 2
Wsig = torch.randn(LAT, RAWDIM, generator=torch.Generator().manual_seed(3)) * 0.7

def make_classes(n, seed):
    g = torch.Generator().manual_seed(seed)
    centers = torch.randn(n, LAT, generator=g)
    sig  = centers[:, None, :].expand(n, PER, LAT) @ Wsig
    jit  = torch.randn(n, PER, LAT, generator=g) * 0.25 @ Wsig
    nuis = torch.randn(n, PER, RAWDIM, generator=g) * 0.7
    return sig + jit + nuis                       # [n, PER, RAWDIM]

train_data = make_classes(80, 1)                  # 80 classes for meta-training
test_data  = make_classes(20, 99)                 # 20 UNSEEN classes for evaluation


# --- 2. The embedding network f_phi (the only thing that learns). ---
def build_embed(seed=0):
    torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(RAWDIM, 64), nn.ReLU(), nn.Linear(64, 16))


# --- 3. Sample an N-way K-shot episode: support + query, fresh classes each call. ---
def sample_episode(data, N, K, Q, gen):
    cls = torch.randperm(data.shape[0], generator=gen)[:N]
    sup, que, qy = [], [], []
    for i, c in enumerate(cls):
        idx = torch.randperm(PER, generator=gen)
        sup.append(data[c, idx[:K]]); que.append(data[c, idx[K:K + Q]]); qy += [i] * Q
    return torch.stack(sup), torch.cat(que), torch.tensor(qy)   # [N,K,RAWDIM],[N*Q,RAWDIM],[N*Q]


# --- 4. The novel algorithm: prototypes (Eqn. 1) + negative-distance logits (Eqn. 2). ---
def proto_logits(emb, sup, que):
    N, K, _ = sup.shape
    proto = emb(sup.reshape(N * K, -1)).reshape(N, K, -1).mean(dim=1)   # Eqn. 1: mean of K shots
    qe = emb(que)
    d = ((qe[:, None, :] - proto[None, :, :])**2).sum(-1)               # squared Euclidean [Nq, N]
    return -d                                                           # Eqn. 2 logits: NEGATIVE d


def eval_acc(emb, data, N=5, K=5, Q=5, episodes=400, seed=123):
    g = torch.Generator().manual_seed(seed); correct = total = 0
    with torch.no_grad():
        for _ in range(episodes):
            sup, que, qy = sample_episode(data, N, K, Q, g)
            correct += int((proto_logits(emb, sup, que).argmax(1) == qy).sum()); total += len(qy)
    return correct / total


# --- 5. Episodic training: each step is a fresh 5-way 5-shot task. ---
N, K, Q = 5, 5, 5
embed_untrained = build_embed(0)
acc_chance_emb  = eval_acc(embed_untrained, test_data)   # random embedding baseline

embed = build_embed(0)
opt = torch.optim.Adam(embed.parameters(), lr=1e-3)
gen = torch.Generator().manual_seed(7)
for step in range(800):
    sup, que, qy = sample_episode(train_data, N, K, Q, gen)
    loss = F.cross_entropy(proto_logits(embed, sup, que), qy)   # J = -log p_phi(y=k|x)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 160 == 0:
        print(f"  step {step:3d}  episode loss {loss.item():.3f}")

acc_trained = eval_acc(embed, test_data)
print(f"\n5-way chance accuracy        : {1/N:.2f}")
print(f"random (untrained) embedding : {acc_chance_emb:.3f}")
print(f"TRAINED prototypical net      : {acc_trained:.3f}   (held-out classes)")
# 5-way chance accuracy        : 0.20
# random (untrained) embedding : ~0.72
# TRAINED prototypical net      : ~0.84   -- far above the 0.20 chance level.
# (Exact numbers vary by hardware; this is our small run, not the paper's reported result.)

## Visualize the data & results

_After episodic training, do query embeddings cluster around their class prototype on held-out (never-trained) classes?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# Reuse the trained 'embed', 'test_data', 'sample_episode' from the CODE panel.
# Embed one held-out 3-way episode, then PCA-project to 2D to SEE the clusters.
with torch.no_grad():
    g = torch.Generator().manual_seed(50)
    sup, que, qy = sample_episode(test_data, 3, 6, 10, g)           # 3 unseen classes
    proto = embed(sup.reshape(18, -1)).reshape(3, 6, -1).mean(1)    # Eqn. 1 prototypes
    qe = embed(que)                                                 # query embeddings

pts = torch.cat([proto, qe], 0)
pts = pts - pts.mean(0)
U, S, V = torch.svd(pts)                # PCA via SVD
proj = pts @ V[:, :2]                   # 16-d -> 2-d
proto2, q2 = proj[:3], proj[3:]
print("prototypes (2D):", [[round(float(a), 2) for a in r] for r in proto2])
for lab in range(3):
    cloud = q2[qy == lab]
    print(f"class {lab} queries (2D):", [[round(float(p[0]), 2), round(float(p[1]), 2)] for p in cloud])
# The three query clouds land in separate regions, each hugging its class prototype.
# (Exact coordinates vary by hardware/seed; our small run, not the paper's number.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The metric ablation. Your trained model classifies queries by softmax over the NEGATIVE
            squared Euclidean distance to each prototype. Replace that with softmax over the positive
            distance (drop the minus sign) and re-evaluate &mdash; no retraining. What happens to accuracy, and
            why does this isolate the role of the negation?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change exactly one character: logits = -d becomes logits = d; keep the embedding, prototypes, data, and everything else identical. — _An honest ablation changes one thing, so any accuracy change is attributable to the sign of the distance in Eqn. 2._
- Re-run the held-out evaluation and watch accuracy fall toward the $1/N$ chance level (about 20% for 5-way). — _With positive distances the softmax now assigns the highest probability to the FARTHEST prototype &mdash; the opposite of "nearest wins."_
- Conclude that the negation, not just the distance, is load-bearing: it is what turns "small distance" into "large probability." — _Eqn. 2 is a softmax over $-d$; remove the minus and the model systematically prefers the wrong class._

**Answer:** Accuracy collapses toward chance (~20% for 5-way). Softmax over $+d$ rewards the prototype
                 that is farthest from the query, which is exactly backwards. This isolates the
                 negation in Eqn. 2 as essential: the model classifies by nearest prototype only
                 because the distance is negated before the softmax. The CODE panel's negated version trains
                 well above chance; this flipped version does not.

</details>

**Problem 2.** A 3-way 4-shot episode. One class's four support examples embed to $[0,0]$, $[2,0]$, $[0,2]$, and
            $[2,2]$. A query for that class embeds to $[1,1]$. Compute the class prototype and the squared
            Euclidean distance from the query to it.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Prototype (Eqn. 1) = mean of the four points: $c = \tfrac{1}{4}([0,0]+[2,0]+[0,2]+[2,2]) = \tfrac{1}{4}[4,4] = [1,1]$. — _Average each coordinate over the $K=4$ shots; the result is one $[1,1]$ prototype for the class._
- Distance from query $[1,1]$ to $c=[1,1]$: $d = (1-1)^2 + (1-1)^2 = 0$. — _Squared Euclidean distance: subtract coordinate-wise, square, sum. Here the query sits exactly on the prototype._
- Note the logit is $-d = 0$, the largest possible for this class, so the query is maximally confident for it (before comparing to the other two prototypes). — _A query landing on its prototype gets the smallest distance and thus the highest pre-softmax score for that class._

**Answer:** The prototype is the mean $[1,1]$, and the query's squared Euclidean distance to it is
                 $d = 0$. The query coincides with its class prototype, so its negative-distance logit $-d = 0$
                 is the maximum &mdash; the strongest possible vote for that class.

</details>

**Problem 3.** In the worked example the query $[3,2]$ got $p(A)=0.9526$ with prototypes $c_A=[2,1]$, $c_B=[5,3]$.
            Now move the query to $q=[4,2]$ (closer to B). Recompute $d_A$, $d_B$, and $p(A)$, and say which
            class wins.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Distances: $d_A = (4-2)^2 + (2-1)^2 = 4+1 = 5$; $d_B = (4-5)^2 + (2-3)^2 = 1+1 = 2$. — _Same squared-Euclidean rule; the query is now nearer B, so $d_B \lt d_A$._
- Logits $-d_A=-5$, $-d_B=-2$; $\exp(-5)=0.006738$, $\exp(-2)=0.135335$. — _Negate the distances and exponentiate, exactly as in Eqn. 2._
- $p(A) = \dfrac{0.006738}{0.006738+0.135335} = 0.0474$, so $p(B)=0.9526$. — _The softmax normalizes the two exponentials; the nearer prototype (B) now takes the large share._

**Answer:** $d_A = 5$, $d_B = 2$, so $p(A) = 0.0474$ and $p(B) = 0.9526$ &mdash; class B wins. The
                 probabilities are exactly the mirror image of the original example, because moving the query
                 from equidistant-favoring-A to closer-to-B swapped which prototype is nearest. Distance to the
                 prototype is the whole story.

</details>